# Mini Project: Public Document RAG API with Gemini

이 노트북은 공개 Wikipedia 문서를 데이터로 사용해 RAG 파이프라인을 만들고, Gemini API로 답변을 생성한 뒤 FastAPI REST API와 RAGAS 평가까지 연결

구현 범위:

1. Document Loading: 공개 Wikipedia 문서 로딩
2. Text Preprocessing: 텍스트 정제
3. Chunking: overlap 기반 청킹
4. Embedding: sentence-transformers 임베딩
5. Vector Store: FAISS 유사도 검색
6. Generation: Gemini 기반 근거 답변 생성
7. Serving: FastAPI REST API
8. Evaluation: RAGAS 평가


In [ ]:
!pip install -q google-genai sentence-transformers faiss-cpu fastapi uvicorn nest_asyncio pyngrok requests beautifulsoup4 pandas datasets ragas langchain-google-genai

In [ ]:
import os
import re
import json
import time
import getpass
import unicodedata
from dataclasses import dataclass
from typing import Any

import faiss
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from google import genai

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY를 입력하세요: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
client = genai.Client(api_key=GEMINI_API_KEY)

GENERATION_MODEL = "gemini-2.5-pro" # 진단 결과에 따라 지원되는 gemini-2.5-pro 모델을 사용함
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

## 1. 공개 문서 로딩

미니프로젝트에서는 공개 Wikipedia 문서를 사용

In [ ]:
WIKI_TITLES = [
    "Retrieval-augmented generation",
    "Large language model",
    "Vector database",
    "Information retrieval",
    "FastAPI",
]


def fetch_wikipedia_extract(title: str) -> dict[str, Any]:
    """Wikipedia MediaWiki API에서 plain text extract를 가져온다."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "redirects": True,
        "titles": title,
    }
    headers = {"User-Agent": "ktb-rag-mini-project/1.0 (educational notebook)"}
    response = requests.get(url, params=params, headers=headers, timeout=20)
    response.raise_for_status()
    pages = response.json()["query"]["pages"]
    page = next(iter(pages.values()))
    return {
        "text": page.get("extract", ""),
        "metadata": {
            "source": "wikipedia",
            "title": page.get("title", title),
            "pageid": page.get("pageid"),
            "url": f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}",
        },
    }


documents = [fetch_wikipedia_extract(title) for title in WIKI_TITLES]
pd.DataFrame([
    {
        "title": doc["metadata"]["title"],
        "chars": len(doc["text"]),
        "url": doc["metadata"]["url"],
    }
    for doc in documents
])


## 2. 전처리와 청킹

RAG 강의 흐름과 동일하게 `Document Loading → Text Preprocessing → Chunking` 순서로 처리

In [ ]:
@dataclass
class Chunk:
    id: str
    text: str
    metadata: dict[str, Any]


def preprocess_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\xad]", "", text)
    text = re.sub(r"={2,}.*?={2,}", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_by_words(text: str, chunk_size: int = 180, chunk_overlap: int = 40) -> list[str]:
    words = text.split()
    chunks = []
    start = 0
    step = chunk_size - chunk_overlap
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks


all_chunks: list[Chunk] = []
for doc_idx, doc in enumerate(documents):
    clean_text = preprocess_text(doc["text"])
    for chunk_idx, chunk_text in enumerate(chunk_by_words(clean_text)):
        all_chunks.append(
            Chunk(
                id=f"doc{doc_idx}-chunk{chunk_idx}",
                text=chunk_text,
                metadata={**doc["metadata"], "chunk_idx": chunk_idx},
            )
        )

print(f"문서 수: {len(documents)}")
print(f"청크 수: {len(all_chunks)}")
print(all_chunks[0])


## 3. 임베딩과 FAISS Vector Store

검색은 cosine similarity와 동일하게 동작하도록 임베딩을 L2 normalize한 뒤 FAISS `IndexFlatIP`를 사용

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [chunk.text for chunk in all_chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("embedding shape:", chunk_embeddings.shape)
print("faiss index size:", index.ntotal)


In [ ]:
def retrieve(question: str, top_k: int = 4) -> list[dict[str, Any]]:
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    scores, indices = index.search(query_embedding, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = all_chunks[int(idx)]
        results.append(
            {
                "chunk_id": chunk.id,
                "score": float(score),
                "text": chunk.text,
                "metadata": chunk.metadata,
            }
        )
    return results


retrieve("What is retrieval augmented generation?", top_k=3)


## 4. Gemini로 답변 생성

프롬프트는 검색된 문서만 근거로 답하도록 제한. 문서에 없는 내용은 모른다고 답하게 하여 환각을 줄임

In [ ]:
def build_prompt(question: str, contexts: list[dict[str, Any]]) -> str:
    context_block = "\n\n".join(
        [
            f"[Document {i + 1}]\n"
            f"title: {ctx['metadata']['title']}\n"
            f"url: {ctx['metadata']['url']}\n"
            f"content: {ctx['text']}"
            for i, ctx in enumerate(contexts)
        ]
    )
    return f"""
You are a RAG question-answering assistant.
Answer in Korean.
Use only the provided documents as evidence.
If the answer is not in the documents, say "제공된 문서에서 해당 정보를 찾을 수 없습니다."
Keep the answer concise, and include source titles at the end.

{context_block}

Question: {question}
Answer:
""".strip()


def generate_answer(question: str, contexts: list[dict[str, Any]]) -> str:
    prompt = build_prompt(question, contexts)
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
    )
    return response.text.strip()


def rag_query(question: str, top_k: int = 4) -> dict[str, Any]:
    contexts = retrieve(question, top_k=top_k)
    answer = generate_answer(question, contexts)
    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "title": ctx["metadata"]["title"],
                "url": ctx["metadata"]["url"],
                "chunk_id": ctx["chunk_id"],
                "score": round(ctx["score"], 4),
            }
            for ctx in contexts
        ],
        "contexts": [ctx["text"] for ctx in contexts],
    }


sample = rag_query("RAG는 LLM 답변을 어떻게 개선하나요?", top_k=4)
print(sample["answer"])
pd.DataFrame(sample["sources"])

## 5. FastAPI REST API로 래핑

백그라운드에서 FastAPI 서버가 올라감
외부에서 접근하려면 `ngrok` 토큰을 설정한 뒤 다음 셀의 주석을 해제 필요

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
import nest_asyncio
import threading
import uvicorn


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1)
    top_k: int = Field(4, ge=1, le=10)


app = FastAPI(title="Public Document RAG API", version="0.1.0")


@app.get("/health")
def health():
    return {"status": "ok", "chunks": len(all_chunks)}


@app.post("/search")
def search_api(request: QueryRequest):
    return {"results": retrieve(request.question, top_k=request.top_k)}


@app.post("/query")
def query_api(request: QueryRequest):
    result = rag_query(request.question, top_k=request.top_k)
    result.pop("contexts", None)
    return result


nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("FastAPI server is running on http://127.0.0.1:8000")


In [ ]:
# 로컬 테스트 진행함
response = requests.post(
    "http://127.0.0.1:8000/query",
    json={"question": "What is a vector database used for in RAG?", "top_k": 4},
    timeout=60,
)
response.json()

In [ ]:
# 선택: ngrok으로 외부 공개 URL 생성함
# 1. https://dashboard.ngrok.com/get-started/your-authtoken 에서 토큰 발급받음
# 2. Colab Secrets에 NGROK_AUTHTOKEN 저장함

# from pyngrok import ngrok
# try:
#     from google.colab import userdata
#     NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
# except Exception:
#     NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN")
#
# if NGROK_AUTHTOKEN:
#     ngrok.set_auth_token(NGROK_AUTHTOKEN)
# public_url = ngrok.connect(8000)
# print(public_url)

## 6. RAGAS 평가

RAGAS 평가는 LLM-as-a-judge 방식이라 Gemini API 호출이 추가로 발생

In [ ]:
eval_questions = [
    {
        "question": "What is retrieval augmented generation?",
        "ground_truth": "Retrieval augmented generation retrieves relevant information from an external knowledge source and uses it to improve a language model's generated answer.",
    },
    {
        "question": "Why are vector databases useful for RAG?",
        "ground_truth": "Vector databases store embeddings and support similarity search, allowing a RAG system to retrieve semantically relevant chunks for a query.",
    },
    {
        "question": "What role does information retrieval play in a RAG system?",
        "ground_truth": "Information retrieval finds relevant documents or passages for the user's query, which are then passed to the generator as context.",
    },
]

eval_rows = []
for item in eval_questions:
    result = rag_query(item["question"], top_k=4)
    eval_rows.append(
        {
            "user_input": item["question"],
            "response": result["answer"],
            "retrieved_contexts": result["contexts"],
            "reference": item["ground_truth"],
        }
    )

eval_df = pd.DataFrame(eval_rows)
eval_df[["user_input", "response", "reference"]]


In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_dataset = EvaluationDataset.from_list(eval_df.to_dict("records"))

evaluator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model=GENERATION_MODEL,
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )
)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="models/text-embedding-004",
        google_api_key=GEMINI_API_KEY,
    )
)

ragas_result = evaluate(
    ragas_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

ragas_result


In [ ]:
ragas_result.to_pandas()